# 📚 Gerador de Dataset para Fine-Tuning (Instrução + Resposta)

Este notebook cria um dataset no formato `question` / `answer` a partir de documentos `.pdf` ou `.txt`.

**Objetivo:** gerar pares de pergunta curta + resposta direta com base no conteúdo do documento, usando um modelo de linguagem local (Qwen2.5-1.5B-Instruct).

**Saída:** arquivo JSONL (uma linha por exemplo), pronto para treinar ou ajustar um modelo RAG ou de QA.

---
## Etapas principais:
1. Instalar dependências necessárias
2. Extrair texto do arquivo (PDF ou TXT)
3. Dividir o texto em pequenos trechos (chunks)
4. Para cada trecho, usar um modelo Hugging Face para gerar uma pergunta e uma resposta curta
5. Salvar os pares gerados em formato JSONL

Vamos começar!


## 1. Instalação das bibliotecas

- `transformers`, `torch`, `accelerate`: para carregar e executar modelos da Hugging Face.
- `pdfplumber`: para extrair texto de PDFs.
- `tqdm`: para mostrar a barra de progresso.

> **Nota:** se você for usar apenas arquivos `.txt`, o `pdfplumber` não é obrigatório, mas vamos mantê-lo para suportar ambos os formatos.

In [ ]:
!pip install transformers torch accelerate pdfplumber tqdm -q

## 2. Importações e configurações

Vamos importar os módulos necessários e desabilitar avisos desnecessários do Transformers para manter a saída limpa.

In [ ]:
import pdfplumber
import json
import torch
from transformers import pipeline, logging
from tqdm import tqdm

# Desativa avisos (apenas erros críticos serão mostrados)
logging.set_verbosity_error()

## 3. Função para extrair texto

Dependendo da extensão do arquivo, chamamos o método adequado:
- **PDF** → `pdfplumber`
- **TXT** → leitura simples com `open`

In [ ]:
def extract_text_from_file(file_path):
    """
    Extrai texto de um arquivo .pdf ou .txt.
    Retorna uma string com todo o conteúdo.
    """
    if file_path.lower().endswith('.pdf'):
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text
    elif file_path.lower().endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    else:
        raise ValueError("Formato não suportado. Use .pdf ou .txt")


## 4. Divisão em trechos (chunks)

Modelos de linguagem têm um limite de tokens. Por isso dividimos o texto em blocos pequenos (ex.: 500 caracteres). Cada bloco será usado para gerar um par pergunta/resposta.

In [ ]:
def split_text(text, max_chunk_length=500):
    """
    Divide o texto em chunks com base em quebras de linha.
    Cada chunk terá no máximo max_chunk_length caracteres.
    """
    paragraphs = text.split("\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) < max_chunk_length:
            current_chunk += para + "\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


## 5. Gerar (Instrução, Resposta) a partir de um trecho

Usamos um modelo da família **Qwen2.5-1.5B-Instruct**, que é leve e muito bom para seguir instruções. O modelo recebe um prompt bem estruturado e deve retornar algo como:

```
INSTRUCTION: O que é X?
RESPONSE: Definição de X.
```

A função abaixo extrai esses campos e retorna a pergunta e a resposta.

In [ ]:
def generate_instruction_response(chunk, hf_pipeline):
    """
    Dado um chunk de texto, usa o pipeline de texto para gerar uma pergunta curta
    e uma resposta direta (ambas com menos de 10 palavras).
    """
    prompt = f"""
You are a data generator for fine-tuning or training a language model.

Given the content below, create:
1. A short and objective question (instruction), under 10 words, based only on the content.
2. A direct and concise answer (response), under 10 words, based only on the content.

Do not add extra explanations. Use the following format:

INSTRUCTION: <short question>
RESPONSE: <short answer>

Content:
\"\"\"
{chunk}
\"\"\"
"""

    messages = [{"role": "user", "content": prompt}]

    try:
        outputs = hf_pipeline(
            messages,
            max_new_tokens=150,
            max_length=None,  # evita warning
            return_full_text=False
        )
        content = outputs[0]["generated_text"]

        # Extrai os campos esperados
        instr_part = content.split("INSTRUCTION:")[1].split("RESPONSE:")[0].strip()
        answer = content.split("RESPONSE:")[1].strip()
        return instr_part, answer
    except Exception as e:
        # Em caso de erro, retorna None, None (o par será ignorado)
        return None, None


## 6. Salvar dataset em formato JSONL

Cada linha do arquivo será um objeto `{"question": "...", "answer": "..."}`. Esse formato é amplamente usado para fine-tuning de modelos de QA.

In [ ]:
def save_to_jsonl(pairs, output_file="dataset.jsonl"):
    """
    pairs: lista de tuplas (instrução, resposta)
    output_file: nome do arquivo de saída
    """
    with open(output_file, "w", encoding="utf-8") as f:
        for instruction, answer in pairs:
            if instruction and answer:
                example = {
                    "Instruction": instruction,
                    "Output": answer
                }
                f.write(json.dumps(example, ensure_ascii=False) + "\n")


## 7. Função principal

A função `generate_dataset` coordena todo o fluxo:
- Carrega o modelo (baixa os pesos se necessário)
- Extrai texto do arquivo (PDF ou TXT)
- Divide em chunks
- Para cada chunk, chama o modelo e coleta os pares
- Salva o resultado

In [ ]:
def generate_dataset(file_path, model_id="Qwen/Qwen2.5-1.5B-Instruct",
                    output_file="dataset.jsonl", max_chunks=None):
    """
    file_path: caminho para o arquivo .pdf ou .txt
    model_id: identificador do modelo no Hugging Face Hub
    output_file: nome do arquivo de saída (JSONL)
    max_chunks: se especificado, limita a quantidade de chunks processados
    """
    print(f"🔄 Carregando modelo: {model_id} ...")

    # Pipeline para geração de texto
    hf_pipeline = pipeline(
        "text-generation",
        model=model_id,
        device_map="auto",        # usa GPU se disponível
        torch_dtype=torch.bfloat16 # reduz consumo de memória
    )

    print("📄 Extraindo texto do arquivo...")
    text = extract_text_from_file(file_path)

    print("✂️  Dividindo em chunks...")
    chunks = split_text(text)

    if max_chunks:
        chunks = chunks[:max_chunks]

    print(f"🧠 Gerando pares (instrução + resposta) para {len(chunks)} chunks...")
    pairs = []

    for chunk in tqdm(chunks, desc="Processando chunks"):
        if len(chunk.strip()) < 10:  # ignora chunks muito curtos
            continue
        instruction, answer = generate_instruction_response(chunk, hf_pipeline)
        if instruction and answer:
            pairs.append((instruction, answer))

    save_to_jsonl(pairs, output_file)
    print(f"\n✅ Dataset salvo em: {output_file} ({len(pairs)} exemplos gerados)")


## 8. Execução

Agora basta informar o caminho do seu arquivo (PDF ou TXT) e, opcionalmente, o número máximo de chunks para testar.

> **Dica:** comece com `max_chunks=5` para verificar se a geração está funcionando. Depois remova o parâmetro ou aumente o valor para processar o documento inteiro.

O modelo será baixado na primeira execução (cerca de 3 GB). Aguarde o download completo.

In [ ]:
# Exemplo de uso com um arquivo .pdf
caminho_arquivo = "manual.pdf"   # Altere para o seu arquivo
# caminho_arquivo = "meu_texto.txt"      # Ou use um .txt

generate_dataset(
    file_path=caminho_arquivo,
    model_id="Qwen/Qwen2.5-1.5B-Instruct",
    output_file="dataset_gerado.jsonl",
    max_chunks=10   # remova esta linha para processar todos os chunks
)

## 🔍 Observações finais

- O modelo utilizado é o **Qwen2.5-1.5B-Instruct**, que funciona bem em CPU ou GPU.
- Para documentos longos, o processo pode levar alguns minutos/horas, dependendo do hardware.
- O dataset gerado pode ser usado para fine-tuning de modelos menores (como BERT, Llama, etc.) ou como base para RAG.
- Se desejar modificar o comprimento das perguntas/respostas, ajuste o `prompt` na função `generate_instruction_response`.

**Divirta-se gerando dados!** 🚀